# 22. Consensus: GS vs SBI Model Assignment

Compares model assignments across four methods:
- GS with update_matrix
- GS with conditional_psych
- SBI with update_matrix
- SBI with conditional_psych

An animal is confidently assigned when all methods agree.


In [ ]:
%matplotlib inline
from shared_setup import *
import pickle
from pathlib import Path


In [ ]:
# ── Mode ──────────────────────────────────────────────────────────────────
# 'load' = load cluster results from results/
# 'run'  = quick local execution (small settings)
MODE = 'load'

if MODE == 'run':
    N_SYNTHETIC = 3
    N_GS_SEEDS = 2
    BURN_IN = 500
    N_SBI_SIMS = 1_000
    N_CV_REPEATS = 4
    SEED = 42
    print('MODE: run (quick local)')
else:
    print('MODE: load (cluster results)')


## 1. Load Results

In [ ]:
CV_DIR = Path('../results/cv')
SBI_DIR = Path('../results/sbi_static')

assignments = {}  # {method_label: {animal: winner}}

if MODE == 'load':
    # GS
    for ft in ['update_matrix', 'conditional_psych']:
        path = CV_DIR / f'uniform_{ft}' / 'summary.pkl'
        if path.exists():
            with open(path, 'rb') as f:
                df = pickle.load(f)['df']
            label = f'GS_{ft[:2].upper()}'
            assignments[label] = dict(zip(df['animal'], df['winner']))
            print(f'Loaded {label}: {len(df)} animals')

    # SBI
    for ft in ['update_matrix', 'conditional_psych']:
        comp_dir = SBI_DIR / 'comparisons' / f'uniform_{ft}'
        if comp_dir.exists():
            files = sorted(comp_dir.glob('animal_*.pkl'))
            d = {}
            for p in files:
                with open(p, 'rb') as f:
                    r = pickle.load(f)
                d[r['animal_id']] = r['winner']
            label = f'SBI_{ft[:2].upper()}'
            assignments[label] = d
            print(f'Loaded {label}: {len(d)} animals')

elif MODE == 'run':
    print('Run notebooks 20 and 21 first, or load cluster results.')


## 2. Consensus Table

In [ ]:
if assignments:
    # Build table: one row per animal, one column per method
    all_animals = sorted(set().union(*[set(d.keys()) for d in assignments.values()]))
    methods = sorted(assignments.keys())

    rows = []
    for aid in all_animals:
        row = {'animal': aid}
        for m in methods:
            row[m] = assignments[m].get(aid, '?')
        # Consensus
        winners = [row[m] for m in methods if row[m] != '?']
        if len(set(winners)) == 1 and len(winners) > 0:
            row['consensus'] = winners[0]
        elif len(winners) == 0:
            row['consensus'] = '?'
        else:
            row['consensus'] = 'SPLIT'
        rows.append(row)

    consensus_df = pd.DataFrame(rows)
    print(consensus_df.to_string(index=False))

    print(f'\nConsensus summary:')
    print(consensus_df['consensus'].value_counts().to_string())
else:
    print('No results to compare.')


## 3. Pairwise Agreement Matrix

In [ ]:
if len(assignments) > 1:
    methods = sorted(assignments.keys())
    n = len(methods)
    agree_matrix = np.zeros((n, n))

    for i, m1 in enumerate(methods):
        for j, m2 in enumerate(methods):
            shared = set(assignments[m1].keys()) & set(assignments[m2].keys())
            if shared:
                agree = sum(assignments[m1][a] == assignments[m2][a] for a in shared)
                agree_matrix[i, j] = agree / len(shared)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(agree_matrix, vmin=0, vmax=1, cmap='RdYlGn')
    ax.set_xticks(range(n))
    ax.set_xticklabels(methods, rotation=45, ha='right')
    ax.set_yticks(range(n))
    ax.set_yticklabels(methods)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{agree_matrix[i,j]:.0%}',
                    ha='center', va='center', fontsize=10)
    ax.set_title('Pairwise Method Agreement')
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()
